# User Preference-Aware Fake News Detection (UPFD)

Graph Classification on UPFD: Hierarchical graph classification modeling news propagation trees and user profiles. This notebook implements the approach with `GCNConv / SAGEConv` inside a `K3UPFDNet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GCNConv / SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "User Preference-Aware Fake News Detection (UPFD)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. UPFD Model Definition
class K3UPFDNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels=2):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, hidden_channels)
        self.lin = layers.Dense(out_channels)

    def call(self, x, edge_index, batch=None):
        x = ops.relu(self.conv1(x, edge_index))
        x = ops.relu(self.conv2(x, edge_index))
        out = k3_layers.global_max_pool(x, batch)
        return self.lin(out)

k3_model = K3UPFDNet(in_channels=128, hidden_channels=64, out_channels=2)

# 2. Forward pass test
num_nodes = 30
dummy_x = keras.random.normal((num_nodes, 128))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")
dummy_batch = ops.zeros((num_nodes,), dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_batch)
print(f"UPFD graph classification prediction shape: {out.shape}")

print("\n✓ K3-Node UPFD execution completed successfully!")